<a href="https://colab.research.google.com/github/jwr231000/Projects/blob/main/Burgers'_8_Layers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
using Random, Statistics, LinearAlgebra, Plots
Random.seed!(42)
u0(x) = -sin(pi*x)
N_epochs = 5000
α = 1e-2
β1 = 0.9
β2 = 0.999
ε = 1e-8
m=64
λ_pde, λ_ic, λ_bc, λ_data = 5.0, 100.0, 10.0, 10.0
x_max, x_min = 1.0, -1.0
t_max, t_min = 1.0, 0.0

N_coll, N_ic, N_bc, N_data = (10000,100,100,20)
x_coll = x_min .+ (x_max-x_min) .* rand(N_coll)
t_coll = t_min .+ (t_max-t_min) .* rand(N_coll)
x_data = x_min .+ (x_max-x_min) .* rand(N_data)
t_data = t_min .+ (t_max-t_min) .* rand(N_data)
x_shock = 0.1 .* randn(30)
t_shock = 0.3 .+ 0.2 .* rand(30)
x_data = vcat(x_data, x_shock)
t_data = vcat(t_data, t_shock)
x_ic = x_min .+ (x_max-x_min) .* rand(N_ic)
t_ic = zeros(N_ic)
u_target_ic = u0.(x_ic)
x_bc = vcat(-ones(div(N_bc, 2)), ones(N_bc - div(N_bc, 2)))
t_bc = t_min .+ (t_max-t_min) .* rand(N_bc)
u_target_bc = zeros(N_bc)
print(1)

In [ ]:
function params(m::Int=64)
W = Vector{Matrix{Float64}}(undef,8)
b = Vector{Vector{Float64}}(undef,8)

W[1] = randn(m, 2) .* sqrt(2/(m+2))
b[1] = zeros(m)
for i in 2:7
W[i] = randn(m, m) .* sqrt(2.0 / m)
b[i] = zeros(m)
end

W[8] = randn(1,m) .* sqrt(2/(1+m))
b[8] = zeros(1)
return W,b
end

In [ ]:
function forward(x,t,W,b)
a = vcat(x', t')
a_cache = [a]
dx_cache = Vector{AbstractArray}(undef, 7)
dt_cache = Vector{AbstractArray}(undef, 7)
z_x_cache = Vector{AbstractArray}(undef, 7)
z_t_cache = Vector{AbstractArray}(undef, 7)
dx = W[1][:, 1]
dt = W[1][:, 2]


for i in 1:7
a = tanh.(W[i] * a .+ b[i])
push!(a_cache, a)
a_squared = 1 .- a.^2
if i == 1
zx = dx
zt = dt
dx = a_squared .* dx
dt = a_squared .* dt

else
zx = W[i] * dx
zt = W[i] * dt
dx = a_squared .* zx
dt = a_squared .* zt
end
z_x_cache[i] = zx
z_t_cache[i] = zt
dx_cache[i] = dx
dt_cache[i] = dt
end
u = vec(W[8] * a .+ b[8])
u_x = vec(W[8] * dx)
u_t = vec(W[8] * dt)
r = u_t .+ u .* u_x
return r, u, u_x, u_t, (a=a_cache, da_dx=dx_cache, da_dt=dt_cache, z_x=z_x_cache, z_t=z_t_cache)
end

In [ ]:
function grad(W,b,x,t,u_target)
N = length(x)
a = vcat(x', t')
a_cache = [a]

for i in 1:7
a = tanh.(W[i] * a .+ b[i])
push!(a_cache, a)
end

u = vec(W[8] * a .+ b[8])
delta = reshape((2).*(u .- u_target) ./ N, 1,N)
dW = Vector{Matrix{Float64}}(undef, 8)
db = Vector{Vector{Float64}}(undef, 8)
dW[8] = delta * a_cache[8]'
db[8] = vec(sum(delta, dims=2))
delta = W[8]' * delta

for i in 7:-1:1
Di = 1 .- a_cache[i+1].^2
delta = delta .* Di
dW[i] = delta * a_cache[i]'
db[i] = vec(sum(delta, dims=2))
delta = W[i]' * delta
end

return dW, db

end

In [ ]:
function grad_pde(u, u_x, u_t, r, cache, W, N_coll)
delta = 2 .* r ./ N_coll
delta_u = delta .* u_x
delta_ut = delta
delta_ux = delta .* u
delta_u = reshape(delta_u, 1, N_coll)
delta_ut = reshape(delta_ut, 1, N_coll)
delta_ux = reshape(delta_ux, 1, N_coll)
a7 = cache.a[8]
a7_x = cache.da_dx[7]
a7_t = cache.da_dt[7]
dw8 = (delta_u*a7') .+ (delta_ux*a7_x') .+ (delta_ut*a7_t')
db8 = [sum(delta_u)]

D7 = 1 .- a7.^2
a6 = cache.a[7]
a6_x = cache.da_dx[6]
a6_t = cache.da_dt[6]
z_x_7 = cache.z_x[7]
z_t_7 = cache.z_t[7]
dw7 = (delta_u.*W[8]'.*D7)*a6' .+
      (delta_ux.*W[8]'.*z_x_7.*(-2).*a7.*D7)*a6' .+
      (delta_ut.*W[8]'.*z_t_7.*(-2).*a7.*D7)*a6' .+
      (delta_ux.*W[8]'.*D7)*a6_x' .+
      (delta_ut.*W[8]'.*D7)*a6_t'

D6 = 1 .- a6.^2
a5 = cache.a[6]
a5_x = cache.da_dx[5]
a5_t = cache.da_dt[5]
z_x_6 = cache.z_x[6]
z_t_6 = cache.z_t[6]
L_6 = W[7]'*(W[8]'.* D7)
D6chain_6 = L_6 .* (-2) .* a6 .* z_x_6 .* D6
D6chain_7 = (W[7]' * (W[8]' .* z_x_7 .* (-2) .* a7 .* D7)) .* D6
D6chain_6_t = L_6 .* (-2) .* a6 .* z_t_6 .* D6
D6chain_7_t = (W[7]' * (W[8]' .* z_t_7 .* (-2) .* a7 .* D7)) .* D6
dw6 = (delta_u.* L_6 .* D6)*a5' .+
      (delta_ux.* L_6 .* D6)*a5_x' .+
      (delta_ux .* D6chain_6)*a5' .+
      (delta_ux .* D6chain_7) * a5' .+


      (delta_ut.* L_6 .* D6)*a5_t' .+
      (delta_ut .* D6chain_6_t)*a5' .+
      (delta_ut .* D6chain_7_t)*a5'

D5 = 1 .- a5.^2
a4 = cache.a[5]
a4_x = cache.da_dx[4]
a4_t = cache.da_dt[4]
z_x_5 = cache.z_x[5]
z_t_5 = cache.z_t[5]
L_5 = W[6]' * (L_6 .* D6)
D5chain_5 = L_5 .* (-2) .* a5 .* z_x_5 .* D5
D5chain_6 = (W[6]' * D6chain_6) .* D5
D5chain_7 = (W[6]' * D6chain_7) .* D5
D5chain_5_t = L_5 .* (-2) .* a5 .* z_t_5 .* D5
D5chain_6_t = (W[6]' * D6chain_6_t) .* D5
D5chain_7_t = (W[6]' * D6chain_7_t) .* D5
dw5 = (delta_u .* L_5 .* D5)*a4' .+
      (delta_ux .* L_5 .* D5)*a4_x' .+
      (delta_ux .* D5chain_5)*a4' .+
      (delta_ux .* D5chain_6)*a4' .+
      (delta_ux .* D5chain_7)*a4' .+

      (delta_ut .* L_5 .* D5)*a4_t' .+
      (delta_ut .* D5chain_5_t)*a4' .+
      (delta_ut .* D5chain_6_t)*a4' .+
      (delta_ut .* D5chain_7_t)*a4'



D4 = 1 .- a4.^2
a3 = cache.a[4]
a3_x = cache.da_dx[3]
a3_t = cache.da_dt[3]
z_x_4 = cache.z_x[4]
z_t_4 = cache.z_t[4]
L_4 = W[5]' * (L_5 .* D5)
D4chain_4 = L_4 .* (-2) .* a4 .* z_x_4 .* D4
D4chain_5 = (W[5]' * D5chain_5) .* D4
D4chain_6 = (W[5]' * D5chain_6) .* D4
D4chain_7 = (W[5]' * D5chain_7) .* D4
D4chain_4_t = L_4 .* (-2) .* a4 .* z_t_4 .* D4
D4chain_5_t = (W[5]' * D5chain_5_t) .* D4
D4chain_6_t = (W[5]' * D5chain_6_t) .* D4
D4chain_7_t = (W[5]' * D5chain_7_t) .* D4
dw4 = (delta_u .* L_4 .* D4)*a3' .+
      (delta_ux .* L_4 .* D4)*a3_x' .+
      (delta_ux .* D4chain_4)*a3' .+
      (delta_ux .* D4chain_5)*a3' .+
      (delta_ux .* D4chain_6)*a3' .+
      (delta_ux .* D4chain_7)*a3' .+

      (delta_ut .* L_4 .* D4)*a3_t' .+
      (delta_ut .* D4chain_4_t)*a3' .+
      (delta_ut .* D4chain_5_t)*a3' .+
      (delta_ut .* D4chain_6_t)*a3' .+
      (delta_ut .* D4chain_7_t)*a3'


D3 = 1 .- a3.^2
a2 = cache.a[3]
a2_x = cache.da_dx[2]
a2_t = cache.da_dt[2]
z_x_3 = cache.z_x[3]
z_t_3 = cache.z_t[3]
L_3 = W[4]' * (L_4 .* D4)
D3chain_3 = L_3 .* (-2) .* a3 .* z_x_3 .* D3
D3chain_4 = (W[4]' * D4chain_4) .* D3
D3chain_5 = (W[4]' * D4chain_5) .* D3
D3chain_6 = (W[4]' * D4chain_6) .* D3
D3chain_7 = (W[4]' * D4chain_7) .* D3
D3chain_3_t = L_3 .* (-2) .* a3 .* z_t_3 .* D3
D3chain_4_t = (W[4]' * D4chain_4_t) .* D3
D3chain_5_t = (W[4]' * D4chain_5_t) .* D3
D3chain_6_t = (W[4]' * D4chain_6_t) .* D3
D3chain_7_t = (W[4]' * D4chain_7_t) .* D3
dw3 = (delta_u .* L_3 .* D3)*a2' .+
      (delta_ux .* L_3 .* D3)*a2_x' .+
      (delta_ux .* D3chain_3)*a2' .+
      (delta_ux .* D3chain_4)*a2' .+
      (delta_ux .* D3chain_5)*a2' .+
      (delta_ux .* D3chain_6)*a2' .+
      (delta_ux .* D3chain_7)*a2' .+

      (delta_ut .* L_3 .* D3)*a2_t' .+
      (delta_ut .* D3chain_3_t)*a2' .+
      (delta_ut .* D3chain_4_t)*a2' .+
      (delta_ut .* D3chain_5_t)*a2' .+
      (delta_ut .* D3chain_6_t)*a2' .+
      (delta_ut .* D3chain_7_t)*a2'


D2 = 1 .- a2.^2
a1 = cache.a[2]
a1_x = cache.da_dx[1]
a1_t = cache.da_dt[1]
z_x_2 = cache.z_x[2]
z_t_2 = cache.z_t[2]
L_2 = W[3]' * (L_3 .* D3)
D2chain_2 = L_2 .* (-2) .* a2 .* z_x_2 .* D2
D2chain_3 = (W[3]' * D3chain_3) .* D2
D2chain_4 = (W[3]' * D3chain_4) .* D2
D2chain_5 = (W[3]' * D3chain_5) .* D2
D2chain_6 = (W[3]' * D3chain_6) .* D2
D2chain_7 = (W[3]' * D3chain_7) .* D2
D2chain_2_t = L_2 .* (-2) .* a2 .* z_t_2 .* D2
D2chain_3_t = (W[3]' * D3chain_3_t) .* D2
D2chain_4_t = (W[3]' * D3chain_4_t) .* D2
D2chain_5_t = (W[3]' * D3chain_5_t) .* D2
D2chain_6_t = (W[3]' * D3chain_6_t) .* D2
D2chain_7_t = (W[3]' * D3chain_7_t) .* D2
dw2 = (delta_u .* L_2 .* D2)*a1' .+
      (delta_ux .* L_2 .* D2)*a1_x' .+
      (delta_ux .* D2chain_2)*a1' .+
      (delta_ux .* D2chain_3)*a1' .+
      (delta_ux .* D2chain_4)*a1' .+
      (delta_ux .* D2chain_5)*a1' .+
      (delta_ux .* D2chain_6)*a1' .+
      (delta_ux .* D2chain_7)*a1' .+

      (delta_ut .* L_2 .* D2)*a1_t' .+
      (delta_ut .* D2chain_2_t)*a1' .+
      (delta_ut .* D2chain_3_t)*a1' .+
      (delta_ut .* D2chain_4_t)*a1' .+
      (delta_ut .* D2chain_5_t)*a1' .+
      (delta_ut .* D2chain_6_t)*a1' .+
      (delta_ut .* D2chain_7_t)*a1'


D1 = 1 .- a1.^2
x_inputs = cache.a[1]
in_x = [ones(N_coll) zeros(N_coll)]
in_t = [zeros(N_coll) ones(N_coll)]
z_x_1 = cache.z_x[1]
z_t_1 = cache.z_t[1]
L_1 = W[2]' * (L_2 .* D2)
D1chain_1 = L_1 .* (-2) .* a1 .* z_x_1 .* D1
D1chain_2 = (W[2]' * D2chain_2) .* D1
D1chain_3 = (W[2]' * D2chain_3) .* D1
D1chain_4 = (W[2]' * D2chain_4) .* D1
D1chain_5 = (W[2]' * D2chain_5) .* D1
D1chain_6 = (W[2]' * D2chain_6) .* D1
D1chain_7 = (W[2]' * D2chain_7) .* D1
D1chain_1_t = L_1 .* (-2) .* a1 .* z_t_1 .* D1
D1chain_2_t = (W[2]' * D2chain_2_t) .* D1
D1chain_3_t = (W[2]' * D2chain_3_t) .* D1
D1chain_4_t = (W[2]' * D2chain_4_t) .* D1
D1chain_5_t = (W[2]' * D2chain_5_t) .* D1
D1chain_6_t = (W[2]' * D2chain_6_t) .* D1
D1chain_7_t = (W[2]' * D2chain_7_t) .* D1
dw1 = (delta_u .* L_1 .* D1)*x_inputs' .+
      (delta_ux .* L_1 .* D1)*in_x .+
      (delta_ux .* D1chain_1)*x_inputs' .+
      (delta_ux .* D1chain_2)*x_inputs' .+
      (delta_ux .* D1chain_3)*x_inputs' .+
      (delta_ux .* D1chain_4)*x_inputs' .+
      (delta_ux .* D1chain_5)*x_inputs' .+
      (delta_ux .* D1chain_6)*x_inputs' .+
      (delta_ux .* D1chain_7)*x_inputs' .+

      (delta_ut .* L_1 .* D1)*in_t .+
      (delta_ut .* D1chain_1_t)*x_inputs' .+
      (delta_ut .* D1chain_2_t)*x_inputs' .+
      (delta_ut .* D1chain_3_t)*x_inputs' .+
      (delta_ut .* D1chain_4_t)*x_inputs' .+
      (delta_ut .* D1chain_5_t)*x_inputs' .+
      (delta_ut .* D1chain_6_t)*x_inputs' .+
      (delta_ut .* D1chain_7_t)*x_inputs'


db8 = [sum(delta_u)]


db7 = vec(sum(delta_u  .* W[8]' .* D7, dims=2)) .+
      vec(sum(delta_ux .* W[8]' .* z_x_7 .* (-2) .* a7 .* D7, dims=2)) .+
      vec(sum(delta_ut .* W[8]' .* z_t_7 .* (-2) .* a7 .* D7, dims=2))

db6 = vec(sum(delta_u  .* L_6 .* D6,dims=2)) .+
      vec(sum(delta_ux .* D6chain_6,dims=2)) .+
      vec(sum(delta_ux .* D6chain_7,dims=2)) .+
      vec(sum(delta_ut .* D6chain_6_t,dims=2)) .+
      vec(sum(delta_ut .* D6chain_7_t,dims=2))

db5 = vec(sum(delta_u  .* L_5 .* D5,dims=2)) .+
      vec(sum(delta_ux .* D5chain_5,dims=2)) .+
      vec(sum(delta_ux .* D5chain_6,dims=2)) .+
      vec(sum(delta_ux .* D5chain_7,dims=2)) .+
      vec(sum(delta_ut .* D5chain_5_t,dims=2)) .+
      vec(sum(delta_ut .* D5chain_6_t,dims=2)) .+
      vec(sum(delta_ut .* D5chain_7_t,dims=2))

db4 = vec(sum(delta_u  .* L_4 .* D4,dims=2)) .+
      vec(sum(delta_ux .* D4chain_4,dims=2)) .+
      vec(sum(delta_ux .* D4chain_5,dims=2)) .+
      vec(sum(delta_ux .* D4chain_6,dims=2)) .+
      vec(sum(delta_ux .* D4chain_7,dims=2)) .+
      vec(sum(delta_ut .* D4chain_4_t,dims=2)) .+
      vec(sum(delta_ut .* D4chain_5_t,dims=2)) .+
      vec(sum(delta_ut .* D4chain_6_t,dims=2)) .+
      vec(sum(delta_ut .* D4chain_7_t,dims=2))

db3 = vec(sum(delta_u  .* L_3 .* D3,dims=2)) .+
      vec(sum(delta_ux .* D3chain_3,dims=2)) .+
      vec(sum(delta_ux .* D3chain_4,dims=2)) .+
      vec(sum(delta_ux .* D3chain_5,dims=2)) .+
      vec(sum(delta_ux .* D3chain_6,dims=2)) .+
      vec(sum(delta_ux .* D3chain_7,dims=2)) .+
      vec(sum(delta_ut .* D3chain_3_t,dims=2)) .+
      vec(sum(delta_ut .* D3chain_4_t,dims=2)) .+
      vec(sum(delta_ut .* D3chain_5_t,dims=2)) .+
      vec(sum(delta_ut .* D3chain_6_t,dims=2)) .+
      vec(sum(delta_ut .* D3chain_7_t,dims=2))

db2 = vec(sum(delta_u  .* L_2 .* D2,dims=2)) .+
      vec(sum(delta_ux .* D2chain_2,dims=2)) .+
      vec(sum(delta_ux .* D2chain_3,dims=2)) .+
      vec(sum(delta_ux .* D2chain_4,dims=2)) .+
      vec(sum(delta_ux .* D2chain_5,dims=2)) .+
      vec(sum(delta_ux .* D2chain_6,dims=2)) .+
      vec(sum(delta_ux .* D2chain_7,dims=2)) .+
      vec(sum(delta_ut .* D2chain_2_t,dims=2)) .+
      vec(sum(delta_ut .* D2chain_3_t,dims=2)) .+
      vec(sum(delta_ut .* D2chain_4_t,dims=2)) .+
      vec(sum(delta_ut .* D2chain_5_t,dims=2)) .+
      vec(sum(delta_ut .* D2chain_6_t,dims=2)) .+
      vec(sum(delta_ut .* D2chain_7_t,dims=2))

db1 = vec(sum(delta_u  .* L_1 .* D1,dims=2)) .+
      vec(sum(delta_ux .* D1chain_1,dims=2)) .+
      vec(sum(delta_ux .* D1chain_2,dims=2)) .+
      vec(sum(delta_ux .* D1chain_3,dims=2)) .+
      vec(sum(delta_ux .* D1chain_4,dims=2)) .+
      vec(sum(delta_ux .* D1chain_5,dims=2)) .+
      vec(sum(delta_ux .* D1chain_6,dims=2)) .+
      vec(sum(delta_ux .* D1chain_7,dims=2)) .+
      vec(sum(delta_ut .* D1chain_1_t,dims=2)) .+
      vec(sum(delta_ut .* D1chain_2_t,dims=2)) .+
      vec(sum(delta_ut .* D1chain_3_t,dims=2)) .+
      vec(sum(delta_ut .* D1chain_4_t,dims=2)) .+
      vec(sum(delta_ut .* D1chain_5_t,dims=2)) .+
      vec(sum(delta_ut .* D1chain_6_t,dims=2)) .+
      vec(sum(delta_ut .* D1chain_7_t,dims=2))

dW_pde = [dw1,dw2,dw3,dw4,dw5,dw6,dw7,dw8]
db_pde = [db1,db2,db3,db4,db5,db6,db7,db8]
return dW_pde, db_pde
end

In [ ]:
function total_grad(u, u_x, u_t, r, cache, W, N_coll, b, x_ic, t_ic, u_target_ic,u_target_bc,u_target_data, λ_pde, λ_ic, λ_bc, λ_data)
dw_pde, db_pde = grad_pde(u, u_x, u_t, r, cache, W, N_coll)
dw_ic,   db_ic   = grad(W, b, x_ic, t_ic, u_target_ic)
dw_bc,   db_bc   = grad(W, b, x_bc, t_bc, u_target_bc)
dw_data, db_data = grad(W, b, x_data, t_data, u_target_data)

dW = [λ_pde*dw_pde[i] + λ_ic*dw_pde[i], λ_bc*dw_bc[i], λ_data*dw_data[i] for i in 1:8]
db = [λ_pde*db_pde[i] + λ_ic*db_pde[i], λ_bc*db_bc[i], λ_data*db_data[i] for i in 1:8]

return dW, db
end

In [ ]:
mutable struct AdamState
m_W::Vector{Matrix{Float64}}
v_W::Vector{Matrix{Float64}}
m_b::Vector{Vector{Float64}}
v_b::Vector{Vector{Float64}}
t::Int
end

function init_adam(W, b)
return AdamState(
        [zeros(size(W[i])) for i in 1:8],
        [zeros(size(W[i])) for i in 1:8],
        [zeros(size(b[i])) for i in 1:8],
        [zeros(size(b[i])) for i in 1:8],
        0)
end

function optim(W, b, dW, db, state::AdamState; α=1e-2, β1=0.9, β2=0.999, ε=1e-8)
    state.t += 1
    bc1 = 1 - β1^state.t
    bc2 = 1 - β2^state.t

    for i in 1:8
        state.m_W[i] .= β1 .* state.m_W[i] .+ (1-β1) .* dW[i]
        state.v_W[i] .= β2 .* state.v_W[i] .+ (1-β2) .* dW[i].^2
        state.m_b[i] .= β1 .* state.m_b[i] .+ (1-β1) .* db[i]
        state.v_b[i] .= β2 .* state.v_b[i] .+ (1-β2) .* db[i].^2

        m_hat_W = state.m_W[i] ./ bc1
        v_hat_W = state.v_W[i] ./ bc2
        m_hat_b = state.m_b[i] ./ bc1
        v_hat_b = state.v_b[i] ./ bc2

        W[i] .-= α .* m_hat_W ./ (sqrt.(v_hat_W) .+ ε)
        b[i] .-= α .* m_hat_b ./ (sqrt.(v_hat_b) .+ ε)
    end
end




In [ ]:
function burgers_reference(; N=400, T=1.0, ν=1e-4, save_dt=0.01)
    dx = 2.0 / (N + 1)
    x_grid = -1.0 .+ dx .* (1:N)
    dt = 0.4 * min(dx, dx^2 / (2ν))
    Nt = ceil(Int, T / dt); dt = T / Nt
    save_every = max(1, round(Int, save_dt / dt))
    u_ref = -sin.(π .* x_grid)
    times = Float64[0.0]
    snapshots = [copy(u_ref)]
    for step in 1:Nt
        u_new = similar(u_ref)
        for i in 1:N
            u_L = (i == 1) ? 0.0 : u_ref[i-1]
            u_R = (i == N) ? 0.0 : u_ref[i+1]
            u_C = u_ref[i]
            conv = u_C >= 0 ? u_C*(u_C - u_L)/dx : u_C*(u_R - u_C)/dx
            diff = ν * (u_R - 2u_C + u_L) / dx^2
            u_new[i] = u_C + dt * (-conv + diff)
        end
        u_ref = u_new
        if step % save_every == 0 || step == Nt
            push!(times, step * dt)
            push!(snapshots, copy(u_ref))
        end
    end
    return x_grid, times, hcat(snapshots...)
end


x_ref, t_ref, U_ref = burgers_reference()

function ref_lookup(x, t, xg, tg, U)
    ix = clamp(searchsortedlast(xg, x), 1, length(xg) - 1)
    it = clamp(searchsortedlast(tg, t), 1, length(tg) - 1)
    fx = (x - xg[ix]) / (xg[ix+1] - xg[ix])
    ft = (t - tg[it]) / (tg[it+1] - tg[it])
    return (1-fx)*(1-ft)*U[ix, it] + fx*(1-ft)*U[ix+1, it] +
           (1-fx)*ft   *U[ix, it+1] + fx*ft   *U[ix+1, it+1]
end

# Convenience wrapper
u_true(x, t) = ref_lookup(x, t, x_ref, t_ref, U_ref)
u_target_data = u_true.(x_data, t_data)    # same length as x_data

In [ ]:
function train!(W, b, state,
                x_coll, t_coll, N_coll,
                x_ic, t_ic, u_target_ic,
                x_bc, t_bc, u_target_bc,
                x_data, t_data, u_target_data,
                N_epochs;
                λ_pde, λ_ic, λ_bc, λ_data,
                α, β1, β2, ε,
                log_every=50)
    losses = Float64[]
    for epoch in 1:N_epochs
        r, u, u_x, u_t, cache = forward(x_coll, t_coll, W, b)
        gW_pde, gb_pde = grad_pde(u, u_x, u_t, r, cache, W, N_coll)
        gW_ic,   gb_ic   = grad(W, b, x_ic,   t_ic,   u_target_ic)
        gW_bc,   gb_bc   = grad(W, b, x_bc,   t_bc,   u_target_bc)
        gW_data, gb_data = grad(W, b, x_data, t_data, u_target_data)

        gW = [λ_pde*gW_pde[i] + λ_ic*gW_ic[i] + λ_bc*gW_bc[i] + λ_data*gW_data[i] for i in 1:8]
        gb = [λ_pde*gb_pde[i] + λ_ic*gb_ic[i] + λ_bc*gb_bc[i] + λ_data*gb_data[i] for i in 1:8]

        optim(W, b, gW, gb, state; α=α, β1=β1, β2=β2, ε=ε)

        if epoch % log_every == 0
            L = mean(r.^2)
            push!(losses, L)
            println("epoch $epoch  L_pde = $(round(L, sigdigits=4))")
        end
    end
    return losses
end

In [ ]:
W, b = params(64)
state = init_adam(W, b)
losses = train!(W, b, state,
                x_coll, t_coll, N_coll,
                x_ic, t_ic, u_target_ic,
                x_bc, t_bc, u_target_bc,
                x_data, t_data, u_target_data,
                N_epochs;
                λ_pde=λ_pde, λ_ic=λ_ic, λ_bc=λ_bc, λ_data=λ_data,
                α=α, β1=β1, β2=β2, ε=ε)

epoch 50  L_pde = 0.2595
epoch 100  L_pde = 0.2572
epoch 150  L_pde = 0.2418
epoch 200  L_pde = 0.2605
epoch 250  L_pde = 0.2381
epoch 300  L_pde = 0.235
epoch 350  L_pde = 0.2339
epoch 400  L_pde = 0.2248
epoch 450  L_pde = 0.2253
epoch 500  L_pde = 0.2416
epoch 550  L_pde = 0.2245
epoch 600  L_pde = 0.2276
epoch 650  L_pde = 0.2204
epoch 700  L_pde = 0.2156
epoch 750  L_pde = 0.2148
epoch 800  L_pde = 0.2111
epoch 850  L_pde = 0.2372
epoch 900  L_pde = 0.2202
epoch 950  L_pde = 0.2081
epoch 1000  L_pde = 0.2018
epoch 1050  L_pde = 0.2159
epoch 1100  L_pde = 0.2099
epoch 1150  L_pde = 0.2179
epoch 1200  L_pde = 0.2322
epoch 1250  L_pde = 0.239
epoch 1300  L_pde = 0.2165
epoch 1350  L_pde = 0.256
epoch 1400  L_pde = 0.2237
epoch 1450  L_pde = 0.2191
epoch 1500  L_pde = 0.2095
epoch 1550  L_pde = 0.2312
epoch 1600  L_pde = 0.2017
epoch 1650  L_pde = 0.2239
epoch 1700  L_pde = 0.2092
epoch 1750  L_pde = 0.2143
epoch 1800  L_pde = 0.2085
epoch 1850  L_pde = 0.2106
epoch 1900  L_pde = 0.23

In [ ]:
using Plots
function plot_loss(losses; log_every=50)
    epochs = log_every .* (1:length(losses))
    p = plot(epochs, losses,
             lw=2, label="PDE loss",
             xlabel="Epoch", ylabel="Loss",
             yscale=:log10,
             title="Training convergence",
             size=(700, 400),
             legend=:topright)
    return p
end

function plot_slices(W, b; times=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
    panels = []
    xs = collect(range(-1, 1, length=300))
    for t_val in times
        ts = fill(t_val, length(xs))
        _, u_p, _, _, _ = forward(xs, ts, W, b)
        u_r = [u_true(x, t_val) for x in xs]
        p = plot(xs, u_r, lw=2.2, label="FD reference", color=:steelblue)
        plot!(p, xs, u_p, lw=1.8, ls=:dash, label="PINN", color=:darkorange)
        title!(p, "t = $(round(t_val, digits=2))")
        xlabel!(p, "x"); ylabel!(p, "u(x,t)")
        ylims!(p, (-1.2, 1.2))
        push!(panels, p)
    end
    return plot(panels..., layout=(3, 4), size=(1400, 900), legend=:topright)
end

function plot_heatmaps(W, b; nx=200, nt=100)
    xs = collect(range(-1, 1, length=nx))
    ts = collect(range(0, 1, length=nt))
    U_pinn = zeros(nx, nt)
    U_ref  = zeros(nx, nt)
    for (j, t_val) in enumerate(ts)
        _, u_p, _, _, _ = forward(xs, fill(t_val, nx), W, b)
        U_pinn[:, j] = u_p
        U_ref[:, j]  = [u_true(x, t_val) for x in xs]
    end
    err = U_pinn .- U_ref

     p_ref  = heatmap(ts, xs, U_ref, color=:RdBu_11, clim=(-1, 1),
                     xlabel="t", ylabel="x", title="FD reference u(x,t)")
    p_pinn = heatmap(ts, xs, U_pinn, color=:RdBu_11, clim=(-1, 1),
                     xlabel="t", ylabel="x", title="PINN prediction u(x,t)")
    p_err  = heatmap(ts, xs, err, color=:bwr, clim=(-0.3, 0.3),
                     xlabel="t", ylabel="x", title="Error (PINN − FD)")

    fig = plot(p_ref, p_pinn, p_err, layout=(1, 3), size=(1400, 400))
    return fig, err
end

function make_all_plots(W, b, losses; log_every=50)
    p_loss = plot_loss(losses; log_every=log_every)
    p_slices = plot_slices(W, b)
    p_maps, err = plot_heatmaps(W, b)

    println("\n========== Final Accuracy ==========")
    println("RMS error of PINN vs FD: $(round(sqrt(mean(err.^2)), digits=4))")
    println("Max absolute error:      $(round(maximum(abs.(err)), digits=4))")

    return p_loss, p_slices, p_maps
end

In [ ]:
p_loss, p_slices, p_maps = make_all_plots(W, b, losses; log_every=50)

display(p_loss)
display(p_slices)
display(p_maps)

